In [73]:
%pip install tsplib95

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import random


def construir_ruta(distancias, feromonas, alfa, beta, rng):
    n = len(distancias)
    inicio = rng.randrange(n)
    
    ruta = [inicio]
    pendientes = list(range(n))
    pendientes.remove(inicio)

    while pendientes:
        actual = ruta[-1]
        pesos = []

        for ciudad in pendientes:
            feromona = feromonas[actual][ciudad]
            cercania = 1.0 / distancias[actual][ciudad]

            peso = (feromona ** alfa) * (cercania ** beta)
            pesos.append(peso)
        
        
        siguiente = rng.choices(
            pendientes,
            weights=pesos,
            k=1
        )[0]

        ruta.append(siguiente)
        pendientes.remove(siguiente)

    ruta.append(inicio)
    return ruta


def calcular_longitud(ruta, distancias):
    total = 0.0

    for paso in range(len(ruta) - 1):
        origen = ruta[paso]
        destino = ruta[paso + 1]

        total += distancias[origen][destino]

    return total


def colonia_hormigas(
    distancias,
    hormigas=20,
    iteraciones=50,
    alfa=1.0,
    beta=2.0,
    evaporacion=0.5,
    Q=100.0,
    semilla=42,
):
    n = len(distancias)
    rng = random.Random(semilla)

    # Inicializar la matriz de feromonas.
    feromonas = []

    for i in range(n):
        fila = []

        for j in range(n):
            fila.append(0.0 if i == j else 1.0)

        feromonas.append(fila)

    mejor_ruta = None
    mejor_longitud = float("inf")
    historial = []

    for _ in range(iteraciones):
        rutas = []
        longitudes = []

        # Construir y evaluar una ruta por hormiga.
        for _ in range(hormigas):
            ruta = construir_ruta(
                distancias, feromonas, alfa, beta, rng
            )

            longitud = calcular_longitud(ruta, distancias)

            rutas.append(ruta)
            longitudes.append(longitud)

            if longitud < mejor_longitud:
                mejor_ruta = ruta.copy()
                mejor_longitud = longitud

        # Evaporar la feromona anterior.
        for i in range(n):
            for j in range(n):
                feromonas[i][j] *= 1.0 - evaporacion

        # Depositar feromonas de todas las hormigas.
        for  ruta_longitud in zip(rutas, longitudes):
            ruta, longitud = ruta_longitud
            deposito = Q / longitud

            for paso in range(len(ruta) - 1):
                i = ruta[paso]
                j = ruta[paso + 1]

                feromonas[i][j] += deposito
                feromonas[j][i] += deposito

        historial.append(mejor_longitud)

    return mejor_ruta, mejor_longitud, historial

In [2]:
import tsplib95

problem = tsplib95.load("kroA100.tsp")

print(problem.name)
print(problem.dimension)
print(problem.edge_weight_type)
nodes = list(problem.get_nodes())

matrix = [
    [
        problem.get_weight(i, j)
        for j in nodes
    ]
    for i in nodes
]

len(matrix)


kroA100
100
EUC_2D


100

In [5]:
nombres = [str(n) for n in nodes]

distancias = matrix

ruta, longitud, historial = colonia_hormigas(distancias, hormigas=10, iteraciones=100)

etiquetas = [nombres[i] for i in ruta]

print("Mejor ruta encontrada:", " -> ".join(etiquetas))
print(f"Distancia total: {longitud:.2f}")
print(
        "Mejor distancia en las primeras 5 iteraciones:",
        historial[:5]
    )

12
38
11
34
6
6
56
75
49
70
13
70
99
28
33
4
24
11
92
46
11
76
90
31
62
14
14
9
69
79
91
18
73
83
17
89
73
23
54
11
2
32
39
5
69
7
75
32
51
4
24
72
72
60
0
72
73
1
76
44
11
77
58
5
95
39
63
78
93
21
93
66
95
69
65
64
65
3
0
58
48
48
78
23
73
78
17
78
51
95
2
66
66
7
41
11
99
99
11
87
87
15
21
5
98
78
58
97
73
72
34
78
64
65
95
6
56
24
54
50
45
85
50
34
50
85
34
95
40
90
50
34
60
97
0
24
86
1
63
72
44
44
86
31
38
89
27
75
9
63
67
41
41
91
96
27
7
78
46
73
89
91
91
92
27
44
22
44
46
16
25
17
89
46
16
55
17
17
12
32
32
38
4
77
16
46
67
38
40
84
51
95
38
67
67
47
25
25
62
15
52
93
65
67
12
27
76
5
76
31
72
39
2
70
80
24
68
73
6
8
56
82
56
40
40
58
47
34
53
44
16
85
26
34
31
73
14
48
7
58
93
23
69
37
98
5
62
72
85
96
88
30
69
7
39
64
3
58
35
0
12
94
2
43
51
51
51
38
85
49
33
94
85
85
43
39
53
72
49
43
43
0
94
31
64
46
25
69
15
46
58
10
31
10
58
0
33
61
2
11
26
61
31
9
73
65
25
37
79
15
80
78
25
62
97
48
18
60
66
96
30
88
91
31
11
82
82
11
22
53
98
72
89
92
50
98
74
93
5
48
10
2
67
3
50
99
5

In [68]:
ruta = [100,45,78]

ruta[-1]

78